In [ ]:
import numpy as np
import os

%load_ext autoreload
%autoreload 2

### Global vars

In [ ]:
import utils_rpc_correction as utils

dataset_dir = "/mnt/adisk/datasets/shadow_dataset"
aoi_id = "JAX_370"
output_dir="tmp_roger" # can be the same as dataset_dir, I just do not have permission
gt_dsm_path = f"tmp_roger/dsm/{aoi_id}/{aoi_id}_resolution_05_2017.tif"
dsm_res = 0.5

print("ok")

### Run bundle adjustment for one AOI

In [ ]:
#utils.run_ba(dataset_dir, aoi_id, output_dir=output_dir)

### Run s2p for one pair

In [ ]:
# part 1 - run s2p for one pair using the bundle-adjusted rpcs
idx1, idx2 = 0, 15
img_path1 = f"{dataset_dir}/crops/{aoi_id}/{aoi_id}_{idx1}.tif"
img_path2 = f"{dataset_dir}/crops/{aoi_id}/{aoi_id}_{idx2}.tif"
json_path1 = f"{output_dir}/root_dir_ba/{aoi_id}/{aoi_id}_{idx1}.json"
json_path2 = f"{output_dir}/root_dir_ba/{aoi_id}/{aoi_id}_{idx2}.json"
img_id1 = os.path.splitext(os.path.basename(json_path1))[0]
img_id2 = os.path.splitext(os.path.basename(json_path2))[0]
s2p_out_dir = os.path.join(output_dir, f"s2p_dsms_ba/{aoi_id}/{img_id1}_{img_id2}")
utils.run_s2p(img_path1, img_path2, json_path1, json_path2, s2p_out_dir, dsm_res)

### [Old strategy] Align ground-truth DSM with the bundle-adjusted RPCs

In [ ]:
# part 2 - align the output dsm with the GT dsm using dsmr
s2p_dsm_path = os.path.join(s2p_out_dir, 'dsm.tif')
gt_rdsm_path = gt_dsm_path.replace("/dsm/", "/rdsm/")
utils.align_gt_dsm(s2p_dsm_path, gt_dsm_path, gt_rdsm_path)

ba_root_dir = f"{output_dir}/root_dir_ba/{aoi_id}"
utils.update_minmax_alt_in_root_dir(gt_rdsm_path, ba_root_dir)

### [New strategy] Compose the bundle adjusted RPCs with the DSM shift

Another strategy would consist in not applying the DSM shift to the GT DSM, but instead compose it with the bundle-adjusted RPCs.

In this case it would not be necessary to update the min and max alt values in the root_dir.

In [ ]:
dsmr_transform_path = gt_rdsm_path.replace(".tif", "_transform.txt")
input_json_dir = f"{output_dir}/root_dir_ba/{aoi_id}"
output_json_dir = f"{output_dir}/root_dir_ba_after_dsmr/{aoi_id}"
utils.update_rpcs_after_dsmr(dsmr_transform_path, input_json_dir, output_json_dir)
utils.update_minmax_alt_in_root_dir(gt_dsm_path, output_json_dir)
print("ok")

### Sanity check

Re-run s2p with the new strategy rpcs. This time the output dsm should be automatically aligned with the ground-truth.

In [ ]:
# ru-run s2p with the new rpcs
idx1, idx2 = 0, 15
img_path1 = f"{dataset_dir}/crops/{aoi_id}/{aoi_id}_{idx1}.tif"
img_path2 = f"{dataset_dir}/crops/{aoi_id}/{aoi_id}_{idx2}.tif"
json_path1 = f"{output_dir}/root_dir_ba_after_dsmr/{aoi_id}/{aoi_id}_{idx1}.json"
json_path2 = f"{output_dir}/root_dir_ba_after_dsmr/{aoi_id}/{aoi_id}_{idx2}.json"
img_id1 = os.path.splitext(os.path.basename(json_path1))[0]
img_id2 = os.path.splitext(os.path.basename(json_path2))[0]
s2p_out_dir = os.path.join(output_dir, f"s2p_dsms_ba_after_dsmr/{aoi_id}/{img_id1}_{img_id2}")
utils.run_s2p(img_path1, img_path2, json_path1, json_path2, s2p_out_dir, dsm_res)
utils.crop_dsm(gt_dsm_path, os.path.join(s2p_out_dir, "dsm.tif"), os.path.join(s2p_out_dir, "dsm_tmp.tif"))

# now these two errors should be ideally the same
s2p_dsm_path = os.path.join(output_dir, f"s2p_dsms_ba/{aoi_id}/{img_id1}_{img_id2}/dsm_tmp.tif")
gt_rdsm_path = gt_dsm_path.replace("/dsm/", "/rdsm/")
minv, maxv = utils.get_minmax_alt_from_dsm(gt_rdsm_path) 
err = utils.err_between_two_aligned_dsms(s2p_dsm_path, gt_rdsm_path, err_path=f"{output_dir}/s1_err.tif", min_alt=minv, max_alt=maxv)
print("error aligning gt with s2p (strategy 1):", np.mean(err))

s2p_dsm_path = os.path.join(output_dir, f"s2p_dsms_ba_after_dsmr/{aoi_id}/{img_id1}_{img_id2}/dsm_tmp.tif")
minv, maxv = utils.get_minmax_alt_from_dsm(gt_dsm_path) 
err = utils.err_between_two_aligned_dsms(s2p_dsm_path, gt_dsm_path, err_path=f"{output_dir}/s2_err.tif", min_alt=minv, max_alt=maxv)
print("error composing ba rpcs with dsmr transform (strategy 2):", np.mean(err))

You can recover the very final jsons with bundle-adjusted rpcs from:

- _output_dir/root_dir_ba_after_dsmr_

The rest of dirs are temporary and could be deleted